# 05 — Multi-Horizon Benchmark: RNN vs LSTM vs GRU
Train each model across horizons [1, 6, 24, 48] with 3 random seeds for statistical robustness.

**Total runs**: 4 horizons x 3 models x 3 seeds = **36 training runs**

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

from gridpulse.data.download_ett import download_ett
from gridpulse.preprocessing.cleaning import clean_ett
from gridpulse.features.feature_builder import build_features
from gridpulse.data.split_time_series import split_by_time
from gridpulse.preprocessing.scaling import ScalerWrapper
from gridpulse.preprocessing.windowing import create_windows, TimeSeriesDataset

from gridpulse.models.rnn import RNNForecaster
from gridpulse.models.lstm import LSTMForecaster
from gridpulse.models.gru import GRUForecaster
from gridpulse.training.trainer import TimeSeriesTrainer
from gridpulse.evaluation.metrics_forecasting import compute_all_metrics
from gridpulse.utils.paths import RAW_DIR
from gridpulse.utils.seed import set_seed

In [2]:
# ── Config ──────────────────────────────────────────────────────────────────
INPUT_LEN    = 96
HORIZONS     = [1, 6, 24, 48]
SEEDS        = [42, 123, 456]
STRIDE       = 1
BATCH_SIZE   = 64
TARGET_COL   = "OT"
DATE_COL     = "date"

HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
DROPOUT      = 0.3

MODEL_CLASSES = [
    ("RNN",  RNNForecaster),
    ("LSTM", LSTMForecaster),
    ("GRU",  GRUForecaster),
]

config = {
    "learning_rate": 0.001,
    "weight_decay": 1e-4,
    "max_epochs": 100,
    "patience": 15,
    "grad_clip": 1.0,
    "batch_size": BATCH_SIZE,
    "input_len": INPUT_LEN,
}

print(f"Horizons: {HORIZONS}")
print(f"Seeds: {SEEDS}")
print(f"Models: {[name for name, _ in MODEL_CLASSES]}")
print(f"Total runs: {len(HORIZONS) * len(MODEL_CLASSES) * len(SEEDS)}")

Horizons: [1, 6, 24, 48]
Seeds: [42, 123, 456]
Models: ['RNN', 'LSTM', 'GRU']
Total runs: 36


## 1. Data Pipeline (shared across all horizons)

In [3]:
download_ett()

ett_path = RAW_DIR / "ett" / "ETTh1.csv"
df_raw = pd.read_csv(ett_path)
df = clean_ett(df_raw)
df_feat = build_features(df, target_col=TARGET_COL, date_col=DATE_COL)

train_df, val_df, test_df = split_by_time(
    df_feat, date_col=DATE_COL, train_ratio=0.6, val_ratio=0.2
)

other_cols = [c for c in df_feat.columns if c not in (DATE_COL, TARGET_COL)]
feature_cols = other_cols + [TARGET_COL]
target_col_idx = -1
NUM_FEATURES = len(feature_cols)

wrapper = ScalerWrapper(scaler=StandardScaler(), columns=feature_cols)
train_scaled = wrapper.fit_transform(train_df)
val_scaled   = wrapper.transform(val_df)
test_scaled  = wrapper.transform(test_df)

train_arr = train_scaled[feature_cols].values
val_arr   = val_scaled[feature_cols].values
test_arr  = test_scaled[feature_cols].values

print(f"Features: {NUM_FEATURES}, Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

2026-07-07 15:28:35.806 | INFO     | gridpulse.data.download_ett:download_ett:20 - Already exists: D:\Project\gridpulse-rnn-timeseries\data\raw\ett\ETTh1.csv
2026-07-07 15:28:35.867 | INFO     | gridpulse.data.download_ett:download_ett:20 - Already exists: D:\Project\gridpulse-rnn-timeseries\data\raw\ett\ETTh2.csv
2026-07-07 15:28:35.871 | INFO     | gridpulse.data.download_ett:download_ett:20 - Already exists: D:\Project\gridpulse-rnn-timeseries\data\raw\ett\ETTm1.csv
2026-07-07 15:28:35.873 | INFO     | gridpulse.data.download_ett:download_ett:20 - Already exists: D:\Project\gridpulse-rnn-timeseries\data\raw\ett\ETTm2.csv
2026-07-07 15:28:36.020 | INFO     | gridpulse.features.feature_builder:build_features:39 - Dropped 168 rows with NaN from feature creations.
2026-07-07 15:28:36.030 | INFO     | gridpulse.data.split_time_series:split_by_time:28 - Split by time: train=10351, val=3450, test=3451


Features: 30, Train: 10351, Val: 3450, Test: 3451


## 2. Train All Models x All Horizons x All Seeds

In [ ]:
all_results = []

for horizon in HORIZONS:
    print(f"\n{'#'*70}")
    print(f"### HORIZON = {horizon}")
    print(f"{'#'*70}")

    # Create windows for this horizon
    X_train, y_train = create_windows(train_arr, INPUT_LEN, horizon, stride=STRIDE, target_col_idx=target_col_idx)
    X_val,   y_val   = create_windows(val_arr,   INPUT_LEN, horizon, stride=STRIDE, target_col_idx=target_col_idx)
    X_test,  y_test  = create_windows(test_arr,  INPUT_LEN, horizon, stride=STRIDE, target_col_idx=target_col_idx)

    print(f"Windows — Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

    train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TimeSeriesDataset(X_val, y_val),     batch_size=64, shuffle=False)
    test_loader  = DataLoader(TimeSeriesDataset(X_test, y_test),   batch_size=64, shuffle=False)

    config["forecast_horizon"] = horizon

    for model_name, model_cls in MODEL_CLASSES:
        seed_metrics = []

        for seed in SEEDS:
            set_seed(seed)

            model = model_cls(
                num_features=NUM_FEATURES,
                hidden_size=HIDDEN_SIZE,
                num_layers=NUM_LAYERS,
                dropout=DROPOUT,
                forecast_horizon=horizon,
            )

            trainer = TimeSeriesTrainer(model, config)
            trainer.fit(train_loader, val_loader, experiment_name=f"dl-benchmark-h{horizon}")

            y_pred = trainer.predict(test_loader)
            metrics = compute_all_metrics(y_test, y_pred)
            metrics["seed"] = seed
            seed_metrics.append(metrics)

            print(f"  {model_name} seed={seed}: MAE={metrics['mae']:.4f} RMSE={metrics['rmse']:.4f}")

        # Aggregate across seeds
        all_results.append({
            "model": model_name,
            "horizon": horizon,
            "mae_mean": round(np.mean([m["mae"] for m in seed_metrics]), 4),
            "mae_std":  round(np.std([m["mae"] for m in seed_metrics]), 4),
            "rmse_mean": round(np.mean([m["rmse"] for m in seed_metrics]), 4),
            "rmse_std":  round(np.std([m["rmse"] for m in seed_metrics]), 4),
            "smape_mean": round(np.mean([m["smape"] for m in seed_metrics]), 2),
            "smape_std":  round(np.std([m["smape"] for m in seed_metrics]), 2),
        })

print(f"\nDone! {len(all_results)} model-horizon combinations.")


######################################################################
### HORIZON = 1
######################################################################


2026-07-07 15:28:36.602 | INFO     | gridpulse.training.trainer:__init__:46 - Using device: cpu
2026-07-07 15:28:36.603 | INFO     | gridpulse.training.trainer:__init__:47 - Model: VanillaRNN_h64_L2, Parameters: 14529


Windows — Train: 10255, Val: 3354, Test: 3355


2026/07/07 15:28:42 INFO mlflow.tracking.fluent: Experiment with name 'dl-benchmark-h1' does not exist. Creating a new experiment.
2026-07-07 15:29:08.748 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 1/100 | Train Loss: 0.091757 | Val Loss: 0.013285 | LR: 1.00e-03
2026-07-07 15:29:08.760 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\VanillaRNN_h64_L2_best.pt (val_loss=0.013285)
2026-07-07 15:29:35.694 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\VanillaRNN_h64_L2_best.pt (val_loss=0.008569)
2026-07-07 15:30:51.037 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 5/100 | Train Loss: 0.031738 | Val Loss: 0.012061 | LR: 1.00e-03
2026-07-07 15:32:31.278 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\VanillaRNN_h6

  RNN seed=42: MAE=0.0583 RMSE=0.0817


...

2026-07-07 16:07:08.762 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 1/100 | Train Loss: 0.127787 | Val Loss: 0.033318 | LR: 1.00e-03
2026-07-07 16:07:08.766 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.033318)
2026-07-07 16:07:29.908 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.028928)
2026-07-07 16:07:50.842 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.027757)
2026-07-07 16:08:11.747 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.019278)
2026-07-07 16:08:32.615 | INFO     | gridpulse.training.trainer

  LSTM seed=42: MAE=0.0671 RMSE=0.0926


2026-07-07 16:17:41.984 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 1/100 | Train Loss: 0.131721 | Val Loss: 0.053757 | LR: 1.00e-03
2026-07-07 16:17:41.990 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.053757)
2026-07-07 16:18:03.384 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.025387)
2026-07-07 16:19:06.337 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 5/100 | Train Loss: 0.032896 | Val Loss: 0.026290 | LR: 1.00e-03
2026-07-07 16:19:27.405 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.013012)
2026-07-07 16:20:52.250 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 10/100 | Train Loss: 0.027425 | Val 

  LSTM seed=123: MAE=0.0632 RMSE=0.0877


2026-07-07 16:30:17.881 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 1/100 | Train Loss: 0.115924 | Val Loss: 0.026701 | LR: 1.00e-03
2026-07-07 16:30:17.886 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.026701)
2026-07-07 16:30:59.578 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.022447)
2026-07-07 16:31:21.473 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gridpulse-rnn-timeseries\models\checkpoints\LSTM_h64_L2_best.pt (val_loss=0.021133)
2026-07-07 16:31:44.378 | INFO     | gridpulse.training.trainer:fit:161 - Epoch 5/100 | Train Loss: 0.034002 | Val Loss: 0.044318 | LR: 1.00e-03
2026-07-07 16:32:30.533 | INFO     | gridpulse.training.callbacks:__call__:46 - Saved best checkpoint: D:\Project\gr

## 3. Results Table

In [ ]:
results_df = pd.DataFrame(all_results)
results_df["mae"] = results_df.apply(lambda r: f"{r['mae_mean']:.4f} ± {r['mae_std']:.4f}", axis=1)
results_df["rmse"] = results_df.apply(lambda r: f"{r['rmse_mean']:.4f} ± {r['rmse_std']:.4f}", axis=1)
results_df["smape"] = results_df.apply(lambda r: f"{r['smape_mean']:.2f} ± {r['smape_std']:.2f}", axis=1)

display_df = results_df[["model", "horizon", "mae", "rmse", "smape"]]
print(display_df.to_markdown(index=False))
display_df

## 4. Pivot Table — MAE by Model x Horizon

In [ ]:
pivot_mae = results_df.pivot(index="model", columns="horizon", values="mae_mean")
pivot_mae = pivot_mae[HORIZONS]
print(pivot_mae.to_markdown())
pivot_mae

## 5. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (metric, label) in zip(axes, [("mae", "MAE"), ("rmse", "RMSE"), ("smape", "sMAPE")]):
    for model_name, _ in MODEL_CLASSES:
        subset = results_df[results_df["model"] == model_name]
        means = subset[f"{metric}_mean"].values
        stds  = subset[f"{metric}_std"].values
        ax.errorbar(HORIZONS, means, yerr=stds, marker="o", capsize=4, label=model_name)

    ax.set_xlabel("Forecast Horizon (hours)")
    ax.set_ylabel(label)
    ax.set_title(f"{label} vs Horizon")
    ax.set_xticks(HORIZONS)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Multi-Horizon Benchmark (mean ± std across 3 seeds)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(HORIZONS))
width = 0.25

for i, (model_name, _) in enumerate(MODEL_CLASSES):
    subset = results_df[results_df["model"] == model_name]
    means = subset["mae_mean"].values
    stds  = subset["mae_std"].values
    bars = ax.bar(x + i * width, means, width, yerr=stds, capsize=3, label=model_name)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xlabel("Forecast Horizon (hours)")
ax.set_ylabel("MAE (mean ± std)")
ax.set_title("MAE Comparison Across Horizons")
ax.set_xticks(x + width)
ax.set_xticklabels([f"H={h}" for h in HORIZONS])
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 6. Best Model per Horizon

In [ ]:
best_per_horizon = results_df.loc[results_df.groupby("horizon")["mae_mean"].idxmin()]
print("Best model per horizon (by MAE):")
print(best_per_horizon[["horizon", "model", "mae", "rmse", "smape"]].to_markdown(index=False))